In [152]:
import os
from dotenv import load_dotenv
import json
import numpy as np
import re
from sklearn.feature_extraction import DictVectorizer
from sentence_transformers import SentenceTransformer
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb

In [2]:
os.chdir("../")

In [ ]:
from etl.transform import *
#from model_training import *

In [ ]:



historical_bucket = os.environ.get("AWS_TRANSFORMED_DATA")

#df = extract_from_s3()

In [ ]:
def train(bucket_name=historical_bucket):
    df = pd.read_csv(f"s3://{bucket_name}/historical_data.csv")

    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    embeddings = embedding_model.encode(df["title"].tolist())

    return df, embeddings

In [42]:
embeddings = train()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [43]:
embeddings.ndim

2

In [12]:
data = train()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
new_data = data.drop(["title", "link"], axis=1)

In [31]:
scaler = MinMaxScaler().set_output(transform="pandas")
df_toscale = new_data.drop(["embeddings", "y"], axis=1)
scaled_data = scaler.fit_transform(df_toscale)

In [34]:
pca = PCA(n_components=3)
principal_components = pca.fit_transform(scaled_data.drop(["price"], axis=1))

In [39]:
embed_np = new_data["embeddings"].to_numpy()
principal_components.ndim

2

In [48]:
price_np = scaled_data["price"].to_numpy()
#X = np.hstack((scaled_data["price"], principal_components, embeddings))
price_np.ndim

1

In [49]:
pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2", "PC3"])

In [54]:
newest = pd.concat([scaled_data["price"], pca_df], axis=1)

In [56]:
newest["embeddings"] = list(embeddings)

In [114]:
newest

,price,PC1,PC2,PC3,embeddings
0,0.032150,-0.358319,-0.323704,0.183798,"[-0.11635679, 0.08875434, 0.053689267, 0.03341..."
1,0.037302,-0.358319,-0.323704,0.183798,"[-0.07805412, -0.010723038, 0.0075451336, -0.0..."
2,0.045031,0.623410,0.081362,-0.001693,"[0.012037751, 0.11134121, 0.020453978, 0.02185..."
3,0.163790,0.623410,0.081362,-0.001693,"[-0.065038696, 0.079567514, 0.027869198, -0.06..."
4,0.024164,0.623410,0.081362,-0.001693,"[-0.09242537, -0.0002480788, 0.04049665, -0.02..."
...,...,...,...,...,...
9205,0.004328,-0.657375,0.680253,0.000782,"[-0.1346935, 0.0031774554, -0.045464247, -0.01..."
9206,0.007934,0.623410,0.081362,-0.001693,"[-0.09806432, 0.0045160293, -0.014932399, 0.01..."
9207,0.007677,-0.363361,-0.372241,0.780533,"[-0.06711594, 0.09736307, 0.05892725, 0.004111..."
9208,0.002782,-0.312397,-0.242093,-0.084609,"[-0.02298641, 0.060917083, 0.015156876, -0.009..."


In [58]:
data_test = data
data_test[["PC1", "PC2", "PC3"]] = list(principal_components)

In [81]:
scaler = MinMaxScaler().set_output(transform="pandas")

new_newdf = data.drop(["title", "link", "PC1", "PC2", "PC3"], axis=1)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(data["title"].tolist())
new_newdf["embeddings"] = list(embeddings)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [82]:
new_newdf

,price,y,acetate,acrylic,cotton,elastane,feathers,glass,leather,linen,...,metal,metallic,modal,other,polyamide,polyester,silk,viscose,wool,embeddings
0,645.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,"[-0.11635679, 0.08875434, 0.053689267, 0.03341..."
1,745.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,"[-0.07805412, -0.010723038, 0.0075451336, -0.0..."
2,895.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,"[0.012037751, 0.11134121, 0.020453978, 0.02185..."
3,3200.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,"[-0.065038696, 0.079567514, 0.027869198, -0.06..."
4,490.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,"[-0.09242537, -0.0002480788, 0.04049665, -0.02..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9205,105.0,0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[-0.1346935, 0.0031774554, -0.045464247, -0.01..."
9206,175.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,"[-0.09806432, 0.0045160293, -0.014932399, 0.01..."
9207,170.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,"[-0.06711594, 0.09736307, 0.05892725, 0.004111..."
9208,75.0,0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,"[-0.02298641, 0.060917083, 0.015156876, -0.009..."


In [83]:
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(new_newdf, new_newdf["y"], test_size=0.3, random_state=42)

In [87]:
pca = PCA(n_components=3)
scaled_data = scaler.fit_transform(X_train_unscaled.drop(["y", "embeddings"], axis=1))
components = pca.fit_transform(scaled_data.drop(["price"], axis=1))

In [143]:
X_train = pd.DataFrame(list(components), columns=["PC1", "PC2", "PC3"])
X_train["price"] = scaled_data["price"].values

X_train

,PC1,PC2,PC3,price
0,0.626688,0.083790,-0.010884,0.032050
1,0.626688,0.083790,-0.010884,0.016077
2,0.626688,0.083790,-0.010884,0.024579
3,0.060514,-0.195391,0.410738,0.023857
4,0.626688,0.083790,-0.010884,0.023548
...,...,...,...,...
6442,0.082321,-0.185887,0.390259,0.015819
6443,-0.383878,-0.148642,0.088534,0.066832
6444,-0.654406,0.681359,0.013137,0.010666
6445,0.626688,0.083790,-0.010884,0.044932


In [ ]:
emb_df = pd.DataFrame(embeddings)
emb_df = emb_df.loc[scaled_data.index]

emb_df.index = range(len(emb_df))
X_train = pd.concat([X_train, emb_df], axis=1)

In [148]:
scaled_test = scaler.transform(X_test_unscaled.drop(["y", "embeddings"], axis=1))
components_test = pca.transform(scaled_test.drop(["price"], axis=1))

X_test = pd.DataFrame(list(components_test), columns=["PC1", "PC2", "PC3"])
X_test["price"] = scaled_test["price"].values

emb_df_ = pd.DataFrame(embeddings)
emb_test = emb_df_.loc[scaled_test.index]

emb_test.index = range(len(emb_test))
X_test = pd.concat([X_test, emb_test], axis=1)

In [146]:
X_train

,PC1,PC2,PC3,price,0,1,2,3,4,5,...,374,375,376,377,378,379,380,381,382,383
0,0.626688,0.083790,-0.010884,0.032050,-0.080956,-0.025083,0.024835,0.028853,0.027644,-0.042967,...,-0.030016,-0.114412,-0.091911,0.055105,-0.030346,0.022798,-0.090562,-0.078339,-0.062610,-0.056247
1,0.626688,0.083790,-0.010884,0.016077,-0.098251,0.041109,0.005293,0.008899,-0.093583,0.044741,...,-0.061229,-0.017309,0.004740,0.037596,0.003996,0.061449,0.004765,-0.102274,0.044035,-0.018749
2,0.626688,0.083790,-0.010884,0.024579,-0.016017,0.032430,-0.045725,0.036414,-0.073380,0.021055,...,0.039985,-0.070569,-0.051818,0.069407,0.019115,0.011874,-0.103557,-0.080917,-0.072111,-0.056807
3,0.060514,-0.195391,0.410738,0.023857,-0.097123,0.029489,0.026394,0.026964,-0.032586,0.009553,...,-0.015121,-0.059196,-0.053939,0.043707,-0.055908,0.006057,-0.074635,-0.076953,-0.054006,-0.036957
4,0.626688,0.083790,-0.010884,0.023548,-0.110451,0.016642,0.039790,0.021959,-0.051369,0.016507,...,-0.068833,0.004805,-0.033154,0.079593,-0.020982,0.070483,0.036662,-0.103083,-0.008162,-0.101287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6442,0.082321,-0.185887,0.390259,0.015819,-0.020348,0.068806,-0.016774,0.001201,0.066388,0.007152,...,-0.046089,0.037966,-0.019440,0.022667,-0.010971,0.082172,0.010019,0.015832,-0.054235,-0.015587
6443,-0.383878,-0.148642,0.088534,0.066832,-0.071027,0.086695,-0.046433,-0.005651,-0.036809,0.015295,...,-0.000831,-0.019417,-0.012342,0.047407,-0.025619,-0.007213,-0.047086,-0.048398,-0.025531,0.036677
6444,-0.654406,0.681359,0.013137,0.010666,-0.057383,0.058464,0.027042,0.008460,-0.035550,0.013031,...,-0.113703,-0.003972,-0.056771,-0.022943,0.033434,0.052351,-0.092261,-0.061706,0.026048,-0.030958
6445,0.626688,0.083790,-0.010884,0.044932,-0.065454,-0.000938,0.030906,0.041476,-0.030856,0.021533,...,-0.018569,-0.058160,-0.012859,0.033270,0.020670,0.024576,-0.085950,-0.055254,-0.041334,-0.040045


In [147]:
model = xgb.XGBClassifier(n_estimators=10)
model.fit(X_train, y_train)


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [149]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

In [151]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, predictions)

array([[2380,   46],
       [ 187,  150]])

In [153]:
f1 = f1_score(y_test, predictions)

In [154]:
f1

0.5628517823639775

In [ ]:
emb_df = pd.DataFrame(embeddings)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:embeddings: object

In [ ]:
print(f"Accuracy: {accuracy * 100:.2f}%")

# Figure out best solution for imbalanced dataset

In [70]:
X_train

,price,PC1,PC2,PC3,embeddings
6829,0.003040,-0.464522,-0.523270,-0.023769,"[-0.03396259, 0.021602605, 0.015721925, 0.0335..."
3161,0.039621,0.755980,0.099012,-0.002207,"[-0.06454255, 0.01812533, 0.08278115, 0.009983..."
4750,0.078778,-0.474806,-0.516871,-0.438545,"[-0.06418872, 0.1479231, 0.038038313, -0.00881..."
1564,0.033953,-0.478863,-0.536627,-0.598631,"[-0.11975244, 0.05740562, 0.015510751, 0.07379..."
1782,0.723839,0.790921,0.111104,-0.002155,"[-0.056127187, 0.027601466, 0.0038693587, 0.04..."
...,...,...,...,...,...
5734,0.015920,0.109262,-0.215279,-0.012106,"[-0.020348312, 0.06880587, -0.016773814, 0.001..."
5191,0.066928,-0.491302,-0.202288,0.192580,"[-0.071026765, 0.08669529, -0.046432678, -0.00..."
5390,0.010768,-0.834012,0.928913,0.000995,"[-0.057383366, 0.05846372, 0.027042035, 0.0084..."
860,0.045031,0.790921,0.111104,-0.002155,"[-0.06545422, -0.00093793956, 0.03090622, 0.04..."


In [ ]:
def fabric_extractor(data):
    fabric_dict = {}
    new_dict = {}

    fiber_abb = {"ac": "acetate", "ca":"acetate", "cmd": "modal", "co": "cotton", "cta": "acetate",
            "cu": "cupro", "cup": "cupro", "cv": "viscose", "ea": "elastane", "el": "elastane",
            "hl": "linen", "li": "linen", "ma": "acrylic", "mo": "modal", "me": "metallic", "ny": "polyamide",
            "pe": "polyester", "pes": "polyester", "pet": "polyester", "pm": "polyester", "pu": "polyester",
            "ra": "ramie", "se": "silk", "ta": "acetate", "vi": "viscose", "wa": "wool", "wg": "wool", "wk": "wool",
            "wl": "wool", "wm": "wool", "wp": "wool", "ws": "wool", "wy": "wool", "wv": "wool", "wo": "wool",
            "wu": "wool", "wb": "wool","pl":"polyester"
            }

    fabric_subs = {"viscose":"viscose", "rayon":"viscose", "spandex":"elastane", "elastane":"elastane", "elastan":"elastane", "tane":"elastane", "alastane":"elastane", "polytrimethylane":"polyester",
               "elasane":"elastane", "elaste":"elastane", "flax":"linen", "linen": "linen", "nylon":"polyamide", "amid":"polyamide", "polia":"polyamide", "terell":"polyester", "elasto":"polyester", 
               "cotton": "cotton", "metal": "metallic", "acetate":"acetate", "modal":"modal", "cupro":"cotton", "modacrylic":"acrylic", "acry": "acrylic","silk":"silk",
               "poly":"polyester", "lurex":"polyester", "wool":"wool", "mohair":"wool", "cashmere":"wool", "merino":"wool", "alpaca":"wool", "seta":"silk", "sisal":"linen",
               "yak":"wool", "angora":"wool", "vicuna":"wool", "llama":"wool", "camel":"wool", "guanaco":"wool", "beaver":"wool", "crepe":"polyester", "satin":"polyester", 
               "korean organza":"polyester", "organza":"silk", "ramie":"linen", "suede":"leather", "leather":"leather", "goose":"feathers", "down":"feathers", "feather":"feathers",
               "bemberg":"cotton", "lycra": "elastane","lyra":"elastane", "acette":"acetate", "ctn":"cotton", "zamac":"metallic", "agnello":"wool", "denim":"cotton",
               "shearling":"leather", "glass":"glass", "polyester":"polyester", "circulose":"cotton", "mesh":"polyester", "lyocell":"lyocell", "tencel":"lyocell", "microtencel":"lyocell",
               "skin":"leather", "pwu":"polyester","lamb":"leather", "creme":"cotton", "elit":"acrylic", "jersey":"polyester","stretch":"polyester","laine":"wool","solvron":"wool",
               "crochet":"cotton","cord":"cotton", "poplin":"cotton","pliss":"polyester","nappa":"leather", "aluminium":"metallic","hemp":"linen","spa":"spandex",
               "brass":"metallic","wax":"cotton","steel":"metal","chaguar":"linen","taffeta":"polyester","econyl":"polyester","poli":"polyester","aluminum":"metallic", "elastan":"elastane", "arcy":"acrylic"
               }

    matches = re.findall(r"(\d+(?:\.\d+)?)%\s*([\w\s-]+?)(?=\d+%|$)", data)

    total_pct = 0
    for pct_string, fabric in matches:
        pct = float(pct_string)
        fabric = fabric.strip()
        if "trochus niloticus" in fabric:
            continue
            
        fabric_dict[fabric] = fabric_dict.get(fabric, 0) + pct
        #total_pct += pct

    for fab_string, pct in fabric_dict.items():

        if fab_string in fiber_abb.keys():
            raw_fiber = fiber_abb[fab_string]
        elif any(key in fab_string for key in fabric_subs):
            for key, val in fabric_subs.items():
                if key in fab_string:
                    raw_fiber = val
                    break
                    
        else:
            raw_fiber = "other"

        new_dict[raw_fiber] = new_dict.get(raw_fiber, 0) + pct
        total_pct += pct


    if total_pct > 100:
        for fabric in new_dict:
            #scaled = (fabric_dict[fabric] / total_pct) * 100
            new_dict[fabric] = int(round((new_dict[fabric] / total_pct) * 100))

    return new_dict

#test2["composition"] = test2["composition"].str.replace(r"[\[\],']|;\s.*|Composition:\s", " ", regex=True)
t = rahrah["composition"].apply(fabric_extractor).tolist()

